# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook explores the FAIR^2 dataset using the `mlcroissant` library for Python.

### Dataset Source
The dataset source is provided as a Croissant schema:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")


## 2. Data Overview

Review available record sets, fields, and their `@id`s using the Croissant metadata.

In [ ]:
# List all record sets and their @id, name, and fields

if dataset.record_sets:
    print("Available record sets:")
    for rs in dataset.record_sets:
        print(f"- Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(no name)')}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print(f"  Fields:")
            for f in fields:
                if isinstance(f, dict):
                    print(f"    - @id: {f.get('@id', '')}, name: {f.get('name', '')}")
                else:
                    print(f"    - {f}")
        print()
else:
    print("No record sets listed in the package metadata. If you expect data, check the Croissant schema or use dataset.to_json() for exploration.")

## 3. Data Extraction

Load data from each record set using `mlcroissant`. Record sets and field IDs are determined from the metadata overview above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []
dataframes = {}

for record_set_id in record_set_ids:
    # Load records into a DataFrame
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set {record_set_id} with {len(records)} records.")
    if len(records) > 0:
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    print()

if not record_set_ids or not dataframes:
    print("No data extracted. See metadata overview above or explore metadata.to_json() for available data structure.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filtering, normalization, grouping, etc.

*Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with the actual `@id`s from your metadata above.*

In [ ]:
# If data is loaded, perform EDA on the first available record set with records.
import numpy as np

if dataframes:
    # Select the first DataFrame with data
    for rsid, df in dataframes.items():
        if not df.empty:
            record_set_id = rsid
            break
    else:
        record_set_id = None

    if record_set_id:
        df = dataframes[record_set_id]
        print(f"Using record set: {record_set_id}\nColumns: {df.columns.tolist()}")

        # Try to find a numeric field
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            print("No numeric columns found for EDA in this record set.")
        else:
            # Pick the first numeric column for demonstrations
            numeric_field_id = numeric_cols[0]
            threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0

            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
            display(filtered_df.head())

            # Normalize the numeric field
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/
                filtered_df[numeric_field_id].std()
            )
            print(f"Normalized example:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by a likely categorical field (try second column)
            possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
            if len(possible_group_fields) > 0:
                group_field_id = possible_group_fields[0]
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print("No categorical/text field for grouping identified.")
else:
    print("No data available for EDA. Ensure you have loaded records into dataframes.")

## 5. Visualization

Visualize data distributions or field relationships using `matplotlib` and `seaborn`. (Optional: adjust based on available fields)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and record_set_id and (len(numeric_cols) > 0):
    # Histogram of first numeric column
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field exists, boxplot
    if len(possible_group_fields) > 0:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[possible_group_fields[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {possible_group_fields[0]}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR^2 Croissant dataset and explored its metadata.
- Identified available record sets and fields using their `@id`s.
- Extracted and previewed records in pandas DataFrames.
- Performed exploratory data analysis and simple visualizations.

Use field and record set `@id`s for reproducibility in analysis and for referencing parts of the dataset programmatically.